In [3]:
import sys
from pathlib import Path
sys.path.append(str((Path.cwd() / ".." / "solvers").resolve()))  # wherever factor_io.py lives

import h5py
import numpy as np
import scipy.sparse as sp
import factor_io as fio

path = "/scratch/yimili/matrices/hdf5/si-bulk.h5"
group = "E_25"
dtype_group = "complex128"

with h5py.File(path, "r") as f:
    g = f[group]

    # --- reconstruct M as CSC (confirmed format), cast to match the factorization dtype ---
    n = len(g["M/indptr"]) - 1
    M = sp.csc_matrix((g["M/data"][:], g["M/indices"][:], g["M/indptr"][:]),
                      shape=(n, n)).astype(np.complex128)

    # --- verify every solver that exposes reconstructable factors ---
    for solver in ("superlu", "umfpack"):
        sg = f.get(f"{group}/{solver}/{dtype_group}")
        if sg is None:
            print(f"{solver:8s}: not present in this group")
            continue
        res  = fio.load_and_verify(sg, M, solver=solver)
        conv = sg.attrs.get("convention", "?")
        status = "OK" if res < 1e-10 else "*** BAD ***"
        print(f"{solver:8s}: max|L U - Pr(...)Pc| = {res:.3e}  {status}   [{conv}]")

import scipy.linalg as sla
from solver_classes import extract_blocks_sparse

path = "/scratch/yimili/matrices/hdf5/graphene.h5"
group = "E_25"
dtype_group = "complex128"

def unpack_lu(lu, piv):
    """Recover A from scipy.linalg.lu_factor output (A = P L U)."""
    m = lu.shape[0]
    L = np.tril(lu, -1) + np.eye(m, dtype=lu.dtype)
    U = np.triu(lu)
    A = L @ U
    for i in range(m - 1, -1, -1):          # undo the forward row swaps
        A[[i, piv[i]]] = A[[piv[i], i]]
    return A

with h5py.File(path, "r") as f:
    g = f[group]
    n = len(g["M/indptr"]) - 1
    M = sp.csc_matrix((g["M/data"][:], g["M/indices"][:], g["M/indptr"][:]),
                      shape=(n, n)).astype(np.complex128)

    bg = f[f"{group}/blockthomas/{dtype_group}"]
    bs      = int(bg.attrs["block_size"])
    L_blk   = bg["L"][:]          # (N-1, bs, bs) subdiagonal blocks (as stored)
    U_blk   = bg["U"][:]          # (N-1, bs, bs) superdiagonal blocks (as stored)
    Dmod_lu = bg["Dmod_lu"][:]    # (N, bs, bs) packed LU of modified diagonal blocks
    Dmod_pv = bg["Dmod_piv"][:]   # (N, bs) pivots

N = Dmod_lu.shape[0]

# block-tridiagonal blocks of the ORIGINAL matrix M
D, L_ext, U_ext = extract_blocks_sparse(M, bs)

# --- (a) is M actually block-tridiagonal at this block size? ---
# (block Thomas silently ignores anything outside the band, so this must hold)
band = sp.lil_matrix((n, n), dtype=M.dtype)
for k in range(N):
    band[k*bs:(k+1)*bs, k*bs:(k+1)*bs] = D[k]
for k in range(N-1):
    band[(k+1)*bs:(k+2)*bs, k*bs:(k+1)*bs] = L_ext[k]
    band[k*bs:(k+1)*bs, (k+1)*bs:(k+2)*bs] = U_ext[k]
off = (M - band.tocsc()); off.eliminate_zeros()
off_norm = np.abs(off.data).max() if off.nnz else 0.0
print(f"(a) max |M outside block-tridiag band| = {off_norm:.3e}   "
      f"({'OK' if off_norm < 1e-10 else '*** M is NOT block-tridiagonal ***'})")

# --- (b) do the stored off-diagonal blocks match M's? ---
dL = max((np.abs(L_blk[k] - L_ext[k]).max() for k in range(N-1)), default=0.0)
dU = max((np.abs(U_blk[k] - U_ext[k]).max() for k in range(N-1)), default=0.0)
print(f"(b) max |stored L - M's L| = {dL:.3e}   max |stored U - M's U| = {dU:.3e}")

# --- (c) the actual factorization invariant ---
worst = 0.0
for k in range(N):
    Dmod_k = unpack_lu(Dmod_lu[k], Dmod_pv[k])          # reconstructed from stored LU
    if k == 0:
        expected = D[0]
    else:
        W = sla.lu_solve((Dmod_lu[k-1], Dmod_pv[k-1]), U_ext[k-1])  # D_mod[k-1]^{-1} U[k-1]
        expected = D[k] - L_ext[k-1] @ W
    worst = max(worst, np.abs(Dmod_k - expected).max())

print(f"(c) max recursion residual over {N} blocks = {worst:.3e}   "
      f"({'OK' if worst < 1e-10 else '*** BAD ***'})")

superlu : max|L U - Pr(...)Pc| = 4.048e-14  OK   [Pr @ A @ Pc == L @ U  (Pr from argsort(perm_r))]
umfpack : max|L U - Pr(...)Pc| = 6.106e-16  OK   [Pr @ diag(1/R) @ A @ Pc == L @ U  (Pr from argsort(perm_r))]
(a) max |M outside block-tridiag band| = 0.000e+00   (OK)
(b) max |stored L - M's L| = 0.000e+00   max |stored U - M's U| = 0.000e+00
(c) max recursion residual over 5 blocks = 4.974e-14   (OK)
